# AlphaFold 2 CPU baseline: reproducible re-run (free Colab)

Regenerates the AF2 CPU (Google Colab CPU runtime) baseline with the benchmark script **as committed in this repository**, instead of a copy pasted into the notebook.

What differs from the original 2026-08-08 run. That notebook, with its outputs, is preserved in git history at commit `faeaa4b`; see `paper/data/canonical_results.md`, discrepancies 11 and 12.

- **Script:** `src/spike_tpu_forward_pass.py` is cloned from the repo at a recorded commit and run with command-line flags: `model_3`, 0 recycles, 118 residues (the toy sequence), float32.
- **Repeats:** 5 steady-state `predict()` calls in the same process. Every value is saved, plus their mean and sample standard deviation. The first, compiling call can only happen once per process.
- **Profiler:** the timed first call runs without `jax.profiler.trace`, whose finalisation inflated the original first-call time. An optional second run with the profiler on measures that overhead on the same runtime.
- **Provenance:** the repo and AlphaFold 2 commits, package versions and hardware details are saved next to the results.
- **Output folder:** everything (environment, pip freeze, logs, result JSONs) is written to `RUN_DIR = results/repro/<YYYY-MM-DD>_cpu-colab/` inside the clone, and the last step downloads it as a zip.

**Before running:** `Runtime > Change runtime type > CPU` (no accelerator). Expect roughly 25 minutes for step 5 on the free CPU runtime (the original run took 42 s for `init_params`, 272 s for the first call and 212 s per steady-state call), plus about 9 minutes for the optional step 6.

## 1. Confirm the runtime has no GPU

In [1]:
import shutil, subprocess

gpu_visible = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi"], capture_output=True).returncode == 0
assert not gpu_visible, (
    "A GPU is attached. This is the CPU baseline: switch to "
    "Runtime > Change runtime type > CPU and run again.")
print("No GPU visible -- correct for the CPU baseline.")

No GPU visible -- correct for the CPU baseline.


## 2. Record the hardware

The original run did not record the CPU model, core count or RAM (`paper/sections/methodology.md`, Section 8).

In [2]:
import json, os, platform, subprocess

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

hardware = {
    "platform": platform.platform(),
    "python": platform.python_version(),
    "cpu_model": run("lscpu | sed -n 's/^Model name:[[:space:]]*//p'"),
    "logical_cpus": os.cpu_count(),
    "mem_total": run("grep MemTotal /proc/meminfo"),
    "lscpu": run("lscpu"),
    "nvidia_smi": None,
}
print(json.dumps({k: v for k, v in hardware.items() if k != "lscpu"}, indent=2))

{
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "python": "3.13.15",
  "cpu_model": "Intel(R) Xeon(R) CPU @ 2.20GHz",
  "logical_cpus": 2,
  "mem_total": "MemTotal:       13286944 kB",
  "nvidia_smi": null
}


## 3. Install dependencies

JAX is pinned to 0.10.2, the version the TPU Jobs in `configs/` install. The other packages are left unpinned, as in the original runs; their resolved versions are recorded in step 7. Colab's preinstalled TensorFlow is used for AlphaFold's feature pipeline and is not reinstalled.

In [3]:
!pip install -q "jax==0.10.2"
!pip install -q dm-haiku ml_collections absl-py biopython numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.4 MB/s eta 0:00:00


## 4. Clone this repository and AlphaFold 2, recording both commits

- **`REPO_REF`** selects the version of `src/spike_tpu_forward_pass.py` to run. It must be an explicit commit hash, not a branch or tag name: replace `PASTE_COMMIT_HASH_HERE` before running, or the cell stops. That commit must include the `--num_steady_state_runs` and `--profile_first_predict` flags; the cell checks this.
- **`ALPHAFOLD_REF = None`** uses AlphaFold 2's default branch, as the original runs did. Set a commit hash to pin it.

In [4]:
import re, shutil, subprocess

REPO_URL = "https://github.com/lorenzopazienza/alphafold-tpu-benchmark.git"
REPO_REF = "9e1a07e453df4d77af15ee695c2cadc1cca73f07"   # explicit commit hash, not a branch name
ALPHAFOLD_URL = "https://github.com/google-deepmind/alphafold.git"
ALPHAFOLD_REF = None    # None = default branch, or a commit hash

assert REPO_REF != "PASTE_COMMIT_HASH_HERE", "Set REPO_REF to the commit hash to run."
assert re.fullmatch(r"[0-9a-f]{7,40}", REPO_REF), (
    f"REPO_REF={REPO_REF!r} is not a commit hash (7-40 lowercase hex characters).")

def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, check=True,
                          capture_output=True, text=True).stdout.strip()

for path in ("/content/bench", "/content/alphafold"):
    shutil.rmtree(path, ignore_errors=True)
git("clone", "-q", REPO_URL, "/content/bench")
git("checkout", "-q", REPO_REF, cwd="/content/bench")
git("clone", "-q", "--depth", "1", ALPHAFOLD_URL, "/content/alphafold")
if ALPHAFOLD_REF:
    git("fetch", "-q", "--depth", "1", "origin", ALPHAFOLD_REF, cwd="/content/alphafold")
    git("checkout", "-q", ALPHAFOLD_REF, cwd="/content/alphafold")

SCRIPT = "/content/bench/src/spike_tpu_forward_pass.py"
missing = [f for f in ("num_steady_state_runs", "profile_first_predict")
           if f'"{f}"' not in open(SCRIPT).read()]
assert not missing, f"REPO_REF={REPO_REF!r} predates the flags {missing}; use a newer ref."

provenance = {
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "repo_commit": git("rev-parse", "HEAD", cwd="/content/bench"),
    "alphafold_url": ALPHAFOLD_URL,
    "alphafold_ref": ALPHAFOLD_REF,
    "alphafold_commit": git("rev-parse", "HEAD", cwd="/content/alphafold"),
}
print(json.dumps(provenance, indent=2))

{
  "repo_url": "https://github.com/lorenzopazienza/alphafold-tpu-benchmark.git",
  "repo_ref": "9e1a07e453df4d77af15ee695c2cadc1cca73f07",
  "repo_commit": "9e1a07e453df4d77af15ee695c2cadc1cca73f07",
  "alphafold_url": "https://github.com/google-deepmind/alphafold.git",
  "alphafold_ref": null,
  "alphafold_commit": "c77e5d2a8961d1a353632c462914ff0a32a950f6"
}


## 4b. Create the run folder and record the environment

Creates `RUN_DIR = /content/bench/results/repro/<YYYY-MM-DD>_cpu-colab/` (UTC date) inside the clone, so its layout matches the repo. It saves:

- **`environment.json`:** UTC timestamp, cloned commit (`git rev-parse HEAD`), CPU model and core counts, full `lscpu`, `free -h`, Python version and kernel.
- **`pip_freeze.txt`:** the packages installed in step 3.

The cell stops if the cloned commit is not `REPO_REF`, or if the clone already has committed files in `RUN_DIR` (a run from the same date).

In [5]:
import datetime, json, os, platform, subprocess, sys

now_utc = datetime.datetime.now(datetime.timezone.utc)
RUN_DIR = f"/content/bench/results/repro/{now_utc:%Y-%m-%d}_cpu-colab"
assert not git("ls-files", RUN_DIR, cwd="/content/bench"), (
    f"{RUN_DIR} already has committed files in the clone; do not overwrite an earlier run.")
os.makedirs(RUN_DIR, exist_ok=True)

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

def lscpu_field(name):
    return sh(f"lscpu | sed -n 's/^{name}:[[:space:]]*//p'")

cloned_commit = git("rev-parse", "HEAD", cwd="/content/bench")
assert cloned_commit.startswith(REPO_REF), (
    f"Cloned commit {cloned_commit} does not match REPO_REF={REPO_REF!r}.")

run_environment = {
    "recorded_at_utc": now_utc.isoformat(timespec="seconds"),
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "git_rev_parse_head": cloned_commit,
    "alphafold_commit": git("rev-parse", "HEAD", cwd="/content/alphafold"),
    "cpu_model": lscpu_field("Model name"),
    "logical_cpus": os.cpu_count(),
    "sockets": lscpu_field("Socket(s)"),
    "cores_per_socket": lscpu_field("Core(s) per socket"),
    "threads_per_core": lscpu_field("Thread(s) per core"),
    "lscpu": sh("lscpu"),
    "free_h": sh("free -h"),
    "python_version": sys.version,
    "kernel": platform.release(),
    "uname_a": sh("uname -a"),
}
with open(f"{RUN_DIR}/environment.json", "w") as f:
    json.dump(run_environment, f, indent=2)
with open(f"{RUN_DIR}/pip_freeze.txt", "w") as f:
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                           capture_output=True, text=True).stdout)

print(f"RUN_DIR:   {RUN_DIR}")
print(f"CPU model: {run_environment['cpu_model']}")
print(f"Cores:     {run_environment['logical_cpus']} logical "
      f"({run_environment['sockets']} socket(s) x {run_environment['cores_per_socket']} core(s) "
      f"x {run_environment['threads_per_core']} thread(s))")
print(f"Commit:    {cloned_commit}")

RUN_DIR:   /content/bench/results/repro/2026-09-17_cpu-colab
CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz
Cores:     2 logical (1 socket(s) x 1 core(s) x 2 thread(s))
Commit:    9e1a07e453df4d77af15ee695c2cadc1cca73f07


## 5. Run the committed script: 5 steady-state calls, profiler off

- **Configuration flags:** `--model_name=model_3 --num_residues=118 --num_recycle=0 --precision=float32`. At 118 residues the script uses its fixed toy sequence (`TOY_SEQUENCE_118`).
- **Timing flags:** `--noprofile_first_predict --num_steady_state_runs=5`.
- **How it runs:** AlphaFold 2 is put on `PYTHONPATH` instead of copying the script into its checkout. The result JSON and the full log are written to `RUN_DIR`.

In [6]:
import json, os, sys

RUN_TAG = "cpu-colab-repro"
RESULTS_DIR = RUN_DIR   # from step 4b
NUM_STEADY_STATE_RUNS = 5
COMMON_FLAGS = "--model_name=model_3 --num_residues=118 --num_recycle=0 --precision=float32"
PY = sys.executable

os.makedirs(RESULTS_DIR, exist_ok=True)
!PYTHONPATH=/content/alphafold {PY} {SCRIPT} --run_tag={RUN_TAG} {COMMON_FLAGS} --noprofile_first_predict --num_steady_state_runs={NUM_STEADY_STATE_RUNS} --results_dir={RESULTS_DIR} 2>&1 | tee {RESULTS_DIR}/log_{RUN_TAG}_noprofile.txt

result_path = f"{RESULTS_DIR}/result_{RUN_TAG}_model_3_len118_recycle0_float32_noprofile.json"
assert os.path.exists(result_path), f"No result JSON at {result_path}: the run failed, see the log above."
result = json.load(open(result_path))
assert result["profile_first_predict"] is False
assert result["num_steady_state_runs"] == NUM_STEADY_STATE_RUNS
assert len(result["steady_state_runs_seconds"]) == NUM_STEADY_STATE_RUNS
print(json.dumps({k: result[k] for k in (
    "init_params_seconds", "first_predict_compile_and_run_seconds",
    "steady_state_runs_seconds", "steady_state_mean_seconds",
    "steady_state_stdev_seconds")}, indent=2))

2026-09-17 08:52:44.135823: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
I0917 08:52:44.195650 135317645439104 xla_bridge.py:849] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0917 08:52:44.196843 135317645439104 spike_tpu_forward_pass.py:155] JAX backend: cpu
I0917 08:52:44.197355 135317645439104 spike_tpu_forward_pass.py:156] JAX devices: [CpuDevice(id=0)]
I0917 08:52:44.197602 135317645439104 spike_tpu_forward_pass.py:157] Run: cpu-colab-repro_model_3_len118_recycle0_float32_noprofile (num_residues=118, num_recycle=0)
I0917 08:52:44.201838 135317645439104 spike_tpu_forward_pass.py:123] Building sequence features for a 118-residue sequence...
I0917 08:52:44.202386 135317645439104 spike_tpu_forward_pass.py:170] Running TF feature-processing pipeline...
I0000 00:00:1789635164.832884 

## 6. Optional: measure the profiler overhead on this runtime (discrepancy 11)

This runs the same configuration with `--profile_first_predict` and one steady-state call, in a separate process, since a process can only compile once.

The difference in `first_predict_compile_and_run_seconds` from step 5 estimates the cost of finalising the profiler trace, which inflated the original baseline. It is one run against one run, so it also contains run-to-run noise. The cost is one more compile; skip this cell if you only need the baseline.

In [7]:
!PYTHONPATH=/content/alphafold {PY} {SCRIPT} --run_tag={RUN_TAG} {COMMON_FLAGS} --profile_first_predict --num_steady_state_runs=1 --results_dir={RESULTS_DIR} 2>&1 | tee {RESULTS_DIR}/log_{RUN_TAG}_profile.txt

profiled_path = f"{RESULTS_DIR}/result_{RUN_TAG}_model_3_len118_recycle0_float32.json"
assert os.path.exists(profiled_path), f"No result JSON at {profiled_path}: the run failed, see the log above."
profiled = json.load(open(profiled_path))
assert profiled["profile_first_predict"] is True
with_prof = profiled["first_predict_compile_and_run_seconds"]
without_prof = result["first_predict_compile_and_run_seconds"]
print(f"First predict, profiler on  (this step): {with_prof:.2f} s")
print(f"First predict, profiler off (step 5):    {without_prof:.2f} s")
print(f"Difference (one run each, includes noise): {with_prof - without_prof:.2f} s")

2026-09-17 09:30:28.592228: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
I0917 09:30:28.670729 138765439189120 xla_bridge.py:849] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0917 09:30:28.671780 138765439189120 spike_tpu_forward_pass.py:155] JAX backend: cpu
I0917 09:30:28.672256 138765439189120 spike_tpu_forward_pass.py:156] JAX devices: [CpuDevice(id=0)]
I0917 09:30:28.672539 138765439189120 spike_tpu_forward_pass.py:157] Run: cpu-colab-repro_model_3_len118_recycle0_float32 (num_residues=118, num_recycle=0)
I0917 09:30:28.676716 138765439189120 spike_tpu_forward_pass.py:123] Building sequence features for a 118-residue sequence...
I0917 09:30:28.677210 138765439189120 spike_tpu_forward_pass.py:170] Running TF feature-processing pipeline...
I0000 00:00:1789637429.203575   12778 ml

## 7. Save provenance

This step writes `environment_<RUN_TAG>.json` and `pip_freeze_<RUN_TAG>.txt` into `RUN_DIR`.

- **`environment_<RUN_TAG>.json`:** commits, hardware, key package versions and flags.
- **`pip_freeze_<RUN_TAG>.txt`:** the full list of installed packages.

In [8]:
import datetime, importlib.metadata, subprocess

PACKAGES = ["jax", "jaxlib", "dm-haiku", "numpy", "tensorflow",
            "ml-collections", "absl-py", "biopython"]
versions = {}
for name in PACKAGES:
    try:
        versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        versions[name] = None

environment = {
    "run_tag": RUN_TAG,
    "recorded_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    **provenance,
    "hardware": hardware,
    "python_executable": PY,
    "package_versions": versions,
    "flags": {"common": COMMON_FLAGS, "num_steady_state_runs": NUM_STEADY_STATE_RUNS},
    "result_files": sorted(f for f in os.listdir(RESULTS_DIR) if f.startswith("result_")),
}
with open(f"{RESULTS_DIR}/environment_{RUN_TAG}.json", "w") as f:
    json.dump(environment, f, indent=2)
with open(f"{RESULTS_DIR}/pip_freeze_{RUN_TAG}.txt", "w") as f:
    f.write(subprocess.run([PY, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
print(json.dumps({k: v for k, v in environment.items() if k != "hardware"}, indent=2))

{
  "run_tag": "cpu-colab-repro",
  "recorded_at_utc": "2026-09-17T09:44:32+00:00",
  "repo_url": "https://github.com/lorenzopazienza/alphafold-tpu-benchmark.git",
  "repo_ref": "9e1a07e453df4d77af15ee695c2cadc1cca73f07",
  "repo_commit": "9e1a07e453df4d77af15ee695c2cadc1cca73f07",
  "alphafold_url": "https://github.com/google-deepmind/alphafold.git",
  "alphafold_ref": null,
  "alphafold_commit": "c77e5d2a8961d1a353632c462914ff0a32a950f6",
  "python_executable": "/usr/bin/python3",
  "package_versions": {
    "jax": "0.10.2",
    "jaxlib": "0.10.2",
    "dm-haiku": "0.0.17",
    "numpy": "2.1.3",
    "tensorflow": "2.20.0",
    "ml-collections": "1.1.0",
    "absl-py": "1.4.0",
    "biopython": "1.88"
  },
  "flags": {
    "common": "--model_name=model_3 --num_residues=118 --num_recycle=0 --precision=float32",
    "num_steady_state_runs": 5
  },
  "result_files": [
    "result_cpu-colab-repro_model_3_len118_recycle0_float32.json",
    "result_cpu-colab-repro_model_3_len118_recycle0_fl

## 8. Zip `RUN_DIR` and download it

Downloads `<YYYY-MM-DD>_cpu-colab.zip`, containing everything in `RUN_DIR`: `environment.json`, `pip_freeze.txt`, the result JSONs, the logs, the step 7 provenance files, and the profiler trace if step 6 ran.

The cell only needs `RUN_DIR`. If the session might expire before the end, it can also be run right after step 5, then again at the end.

Unzip it into `results/repro/` in the repo.

In [9]:
import os, shutil
from google.colab import files

archive = shutil.make_archive(f"/content/{os.path.basename(RUN_DIR)}", "zip",
                              root_dir=os.path.dirname(RUN_DIR),
                              base_dir=os.path.basename(RUN_DIR))
print(f"{archive}: {os.path.getsize(archive) / 1e6:.1f} MB")
for name in sorted(os.listdir(RUN_DIR)):
    print("  ", name)
files.download(archive)

/content/2026-09-17_cpu-colab.zip: 92.1 MB
   environment.json
   environment_cpu-colab-repro.json
   log_cpu-colab-repro_noprofile.txt
   log_cpu-colab-repro_profile.txt
   pip_freeze.txt
   pip_freeze_cpu-colab-repro.txt
   result_cpu-colab-repro_model_3_len118_recycle0_float32.json
   result_cpu-colab-repro_model_3_len118_recycle0_float32_noprofile.json
   trace_cpu-colab-repro_model_3_len118_recycle0_float32


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>